<a href="https://colab.research.google.com/github/Baze-Bai/LLM/blob/HW2/Transformer_structure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
impoort numpy

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
  '''
  Compute scaled dot product attention
  Q: query shape: (batch_size, num_heads, seq_len, depth)
  K: Key vector (key) shape: (batch_size, num_heads, seq_len, depth)
  V: value vector (value) shape: (batch_size, num_heads, seq_len, depth)
  Masking Mask for masking values at specific locations (optional)
  '''
  d_k = Q.shape[-1] # Caculate the depth of Q, which is used to scale the denominator of A to prevent the gradient disappearance or explosion

  # Compute the dot product of Q and K, and scale it
  scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(d_k)

  # If a mask exists, apply it to the fractional matrix
  if mask is not None:
    scores += (mask * -1e9)

  # Perform softmax on the score matrix
  attention_weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True)) # Subtract the maximum value to prevent overflow
  attention_weights /= np.sum(attention_weights, axis=-1, keepdims=True)

  # Compute the output of the attention mechanism
  output = np.matmul(attention_weights, V)

  return output

# Define the multi attention module
Class MultiHeadAttention:
  def __init__(self, d_model, num_heads):
    '''
    d_model: depth of the model
    num_heads: number of heads
    '''
    assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
    self.d_model = d_model
    self.num_heads = num_heads

    # Initialize the weighting matrix
    self.W_q = np.random.randn(d_model, d_model)
    self.W_k = np.random.randn(d_model, d_model)
    self.W_v = np.random.randn(d_model, d_model)
    self.W_o = np.random.randn(d_model, d_model)

  def split_heads(self, x):

    batch_size, seq_len, d_model = x.shape
    x = x.reshape(batch_size, seq_len, self.num_heads, self.depth)
    return x.transpose(0, 2, 1, 3)

  def _call_(self, Q, K, V, mask=None):
    # Calculate the projection of Q, K, and V
    Q = np.matmul(Q, self.W_q)
    K = np.matmul(K, self.W_k)
    V = np.matmul(V, self.W_v)

    # Split into multiple
    Q = self.split_heads(Q)
    K = self.split_heads(K)
    V = self.split_heads(V)

    # Calculate the attention scores
    attention = scaled_dot_product_attention(Q, K, V, mask)

    # Merge Multiple Outputs
    attention = attention.transpose(0, 2, 1, 3).reshape(Q.shape[0], -1, self.num_heads * self.depth)

    # Calculate the final output
    output = np.matmul(attention, self.W_o)

    return output

Class FeedForward:
  def __init__(self, d_model, d_ff):
    self.W_1 = np.random.randn(d_model, d_ff)
    self.W_2 = np.random.randn(d_ff, d_model)
    self.b_1 = np.random.randn(d_ff)
    self.b_2 = np.random.randn(d_model)

  def _call_(self, x):
    # Two-tier fully connected layer
    x = np.maximum(0, np.matmul(x, self.W_1) + self.b_1)
    x = np.matmul(x, self.W_2) + self.b_2
    return x

  def positional_encoding(seq_len, d_model):
    # The index of each position
    position = np.arange(seq_len)[:, np.newaxis]
    # scaling factor
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(position * div_term) # sin for even position
    pe[:, 1::2] = np.cos(position * div_term) # cos for odd position
    return pe

Class EncoderLayer:
  def __init__(self, d_model, num_heads, d_ff):
    # The multi attention head module
    self.mha = MultiHeadAttention(d_model, num_heads)
    # The feed forward module
    self.ff = FeedForward(d_model, d_ff)
    # Layer normalization
    self.norm1 = lambda x: (x - np.mean(x, axis=-1, keepdims=True)) / (np.std(x, axis=-1, keepdims=True) + 1e-6) # Noramlization for the first layer
    self.norm2 = lambda x: (x - np.mean(x, axis=-1, keepdims=True)) / (np.std(x, axis=-1, keepdims=True) + 1e-6) # Normalization for the second layer

  def _call_(self, x, mask=None):
    # Multi-head attention residual connection layernorm
    attn_output = self.mha(x, x, x, mask)
    out1 = self.layernorm1(x + attn_output)

    # Feed forward residual connection and layernorm
    ff_output = self.ff(out1)
    out2 = self.layernorm2(out1 + ff_output)

    return out2

# Transformer Decoder Layer
class DecoderLayer:
    def __init__(self, d_model, num_heads, d_ff):
        self.mha1 = MultiHeadAttention(d_model, num_heads)  # self attetion module
        self.mha2 = MultiHeadAttention(d_model, num_heads)  # Encoder-Decoder Attention Module
        self.ffn = FeedForward(d_model, d_ff)  # Feed forward neural network
        self.layernorm1 = lambda x: (x - np.mean(x, axis=-1, keepdims=True)) / (np.std(x, axis=-1, keepdims=True) + 1e-6)
        self.layernorm2 = lambda x: (x - np.mean(x, axis=-1, keepdims=True)) / (np.std(x, axis=-1, keepdims=True) + 1e-6)
        self.layernorm3 = lambda x: (x - np.mean(x, axis=-1, keepdims=True)) / (np.std(x, axis=-1, keepdims=True) + 1e-6)

    def __call__(self, x, enc_output, look_ahead_mask, padding_mask):
        # First Layer Self-Attention + Residual Connection + LayerNorm
        attn1 = self.mha1(x, x, x, look_ahead_mask)  # Decoder Self Attention
        out1 = self.layernorm1(x + attn1)  # residual linkage post-normalization

        # Layer 2 Encoder-Decoder Attention + Residual Connection + LayerNorm
        attn2 = self.mha2(out1, enc_output, enc_output, padding_mask)  # spanning attention
        out2 = self.layernorm2(out1 + attn2)

        # Feedforward Network + Residual Connection + LayerNorm
        ffn_output = self.ffn(out2)  # feed-forward network
        out3 = self.layernorm3(out2 + ffn_output)

        return out3

# Define the complete Transformer Decoder
class TransformerDecoder:
  def __init__(self, num_layers, d_model, num_heads, d_ff, target_vocab_size, max_seq_len):
      self.num_layers = num_layers
      self.d_model = d_model
      self.embedding = np.random.randn(target_vocab_size, d_model)  # Target vocabulary embedding matrix
      self.pos_encoding = positional_encoding(max_seq_len, d_model)  # Positional encoding
      self.dec_layers = [DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)]  # Multiple decoder layers

  def __call__(self, x, enc_output, look_ahead_mask, padding_mask):
      seq_len = x.shape[1]

      # Embedding + Positional Encoding
      x = np.matmul(x, self.embedding) + self.pos_encoding[:seq_len, :]  # Add positional encoding to word embeddings

      # Pass through each decoder layer
      for i in range(self.num_layers):
          x = self.dec_layers[i](x, enc_output, look_ahead_mask, padding_mask)

      return x

# Example hyperparameter configuration
num_layers = 2
d_model = 64
num_heads = 8
d_ff = 256
input_vocab_size = 10000
target_vocab_size = 10000
max_seq_len = 100

# Initialize Transformer encoder and decoder
encoder = TransformerEncoder(num_layers, d_model, num_heads, d_ff, input_vocab_size, max_seq_len)
decoder = TransformerDecoder(num_layers, d_model, num_heads, d_ff, target_vocab_size, max_seq_len)

# Example input (assuming input is one-hot encoded word indices)
batch_size = 32
seq_len = 20
x = np.random.randint(0, input_vocab_size, (batch_size, seq_len))
target_seq = np.random.randint(0, target_vocab_size, (batch_size, seq_len))
look_ahead_mask = None  # Look-ahead mask for self-attention
padding_mask = None  # Padding mask for cross-attention

# Encoder output
enc_output = encoder(x, padding_mask)

# Decoder output
dec_output = decoder(target_seq, enc_output, look_ahead_mask, padding_mask)
print("Transformer Decoder Output Shape:", dec_output.shape)
